Robustness Checks

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.manifold import TSNE, trustworthiness
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy.spatial import procrustes
import umap

from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
from datasets import load_dataset, Audio, DatasetDict

SEED_LIST = [0, 1, 2, 3, 4]
K_NEIGHBORS = 10
EMOTIONS = ["angry", "disgust", "fear", "happy", "neutral", "pleasant surprise", "sad"]
COLORS = {e: plt.cm.tab10(i) for i, e in enumerate(EMOTIONS)}   # ίδιο χρώμα ανά συναίσθημα

## 1. Φόρτωση μοντέλου, feature extractor και dataset

In [ ]:
FT_REPO = "VasilisAsim/data2vec-finetuned-TESS_full"
model_ft = AutoModelForAudioClassification.from_pretrained(FT_REPO)
feature_extractor_ft = AutoFeatureExtractor.from_pretrained(FT_REPO)
model_ft.eval();

In [ ]:
def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        filename = example["audio"]["path"]
        return "OAF" if "OAF" in filename else ("YAF" if "YAF" in filename else "unknown")

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")
    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    return DatasetDict({"train": train_dataset, "test": test_dataset})

dataset = load_tess_speaker_independent()
test_dataset = dataset["test"]

In [ ]:
#Συνάρτηση εξαγωγής των αναπαραστάσεων του embedding space που δημιουργεί το μοντέλο
def extract_embeddings(model, feature_extractor, dataset):
    embeddings = []
    labels = []

    model.eval()
    for sample in dataset:
        audio = sample["audio"]["array"]
        label = sample["emotion"]

        #Προεπεξεργασία κάθε στοιχείου του dataset
        inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt")

        #Εξαγωγή των αναπαραστάσεων που δημιοργεί το μοντέλο
        with torch.no_grad():
            outputs = model.data2vec_audio(inputs["input_values"])

        #Δημιουργία ενιαίου vector από το στοιχείο
        pooled = outputs.last_hidden_state.mean(dim=1)
        embeddings.append(pooled.squeeze().numpy())
        labels.append(label)

    return np.array(embeddings), np.array(labels)

In [ ]:
def prepare(emb):
    """Κεντράρισμα και L2-κανονικοποίηση, ώστε η ευκλείδεια απόσταση να είναι ισοδύναμη με την cosine."""
    emb = emb.astype(np.float64)
    emb = emb - emb.mean(axis=0, keepdims=True)
    return emb / np.linalg.norm(emb, axis=1, keepdims=True)


embeddings, labels = extract_embeddings(model_ft, feature_extractor_ft, test_dataset)
embeddings = prepare(embeddings)
print(embeddings.shape, labels.shape)

## 2. Συναρτήσεις μετρικών robustness

In [ ]:
def knn_overlap(X_high, X_low, k=10):
    """
    Μέσο ποσοστό επικάλυψης k-πλησιέστερων γειτόνων ανάμεσα σε αρχικό και προβεβλημένο χώρο.
    """
    nn_high = NearestNeighbors(n_neighbors=k + 1).fit(X_high)
    nn_low = NearestNeighbors(n_neighbors=k + 1).fit(X_low)
    idx_high = nn_high.kneighbors(return_distance=False)[:, 1:]
    idx_low = nn_low.kneighbors(return_distance=False)[:, 1:]
    overlaps = [
        len(set(idx_high[i]) & set(idx_low[i])) / k
        for i in range(X_high.shape[0])
    ]
    return float(np.mean(overlaps))


def continuity(X_high, X_low, k=10):
    """
    Αντίστροφο της trustworthiness: αν οι πραγματικοί γείτονες παραμένουν γείτονες στο 2D.
    Χρησιμοποιεί την ίδια rank-based φόρμουλα με αντεστραμμένα ορίσματα.
    """
    return float(trustworthiness(X_low, X_high, n_neighbors=k))


def procrustes_disparity(emb_a, emb_b):
    """
    Disparity μετά από Procrustes ευθυγράμμιση (rotation/scale/translation).
    Χαμηλότερο = πιο σταθερό σχήμα.
    """
    _, _, disparity = procrustes(emb_a, emb_b)
    return float(disparity)

## 3. Κύρια συνάρτηση robustness check ανά μέθοδο

In [ ]:
def make_reducer(method, seed, init="default"):
    """
    init="default": προεπιλεγμένη αρχικοποίηση (t-SNE: PCA, UMAP: spectral).
    init="random": τυχαία αρχικοποίηση, αυστηρότερος έλεγχος ως προς το seed.
    """
    if method == "tsne":
        return TSNE(n_components=2, perplexity=30, random_state=seed,
                    init="pca" if init == "default" else "random")
    elif method == "umap":
        return umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric="cosine",
                         init="spectral" if init == "default" else "random", random_state=seed)
    elif method == "pca":
        return PCA(n_components=2, svd_solver="randomized", random_state=seed)
    raise ValueError(method)


def robustness_check(embeddings, labels, method, init="default", seeds=SEED_LIST, k=K_NEIGHBORS):
    name = method if method == "pca" else f"{method}-{init}"
    n_clusters = len(np.unique(labels))
    per_seed_rows = []
    projections, cluster_assignments = [], []

    for seed in seeds:
        emb_2d = make_reducer(method, seed, init).fit_transform(embeddings)
        projections.append(emb_2d)

        trust = trustworthiness(embeddings, emb_2d, n_neighbors=k)
        cont = continuity(embeddings, emb_2d, k=k)
        overlap = knn_overlap(embeddings, emb_2d, k=k)

        km = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10).fit(emb_2d)
        cluster_assignments.append(km.labels_)
        per_seed_rows.append({
            "method": name, "seed": seed,
            "trustworthiness": trust, "continuity": cont, "knn_overlap": overlap,
            "ari_vs_true_labels": adjusted_rand_score(labels, km.labels_),
            "nmi_vs_true_labels": normalized_mutual_info_score(labels, km.labels_),
        })

    ref_proj = projections[0]
    procrustes_scores = [procrustes_disparity(ref_proj, p) for p in projections[1:]]
    ref_clusters = cluster_assignments[0]
    ari_between = [adjusted_rand_score(ref_clusters, c) for c in cluster_assignments[1:]]
    nmi_between = [normalized_mutual_info_score(ref_clusters, c) for c in cluster_assignments[1:]]

    df = pd.DataFrame(per_seed_rows)
    summary = {
        "method": name,
        "trustworthiness_mean": df["trustworthiness"].mean(), "trustworthiness_std": df["trustworthiness"].std(),
        "continuity_mean": df["continuity"].mean(), "continuity_std": df["continuity"].std(),
        "knn_overlap_mean": df["knn_overlap"].mean(), "knn_overlap_std": df["knn_overlap"].std(),
        "procrustes_disparity_mean": np.mean(procrustes_scores), "procrustes_disparity_std": np.std(procrustes_scores),
        "ari_between_seeds_mean": np.mean(ari_between), "nmi_between_seeds_mean": np.mean(nmi_between),
        "ari_vs_true_mean": df["ari_vs_true_labels"].mean(), "nmi_vs_true_mean": df["nmi_vs_true_labels"].mean(),
    }
    return df, summary, projections

## 4. Εκτέλεση για t-SNE, UMAP, PCA

In [ ]:
VARIANTS = [("tsne", "default"), ("tsne", "random"),
            ("umap", "default"), ("umap", "random"),
            ("pca", "default")]

all_dfs, all_summaries, all_projections = [], [], {}
for method, init in VARIANTS:
    name = method if method == "pca" else f"{method}-{init}"
    print(f"Running robustness check for: {name}")
    df, summary, projections = robustness_check(embeddings, labels, method, init=init)
    all_dfs.append(df)
    all_summaries.append(summary)
    all_projections[name] = projections

per_seed_results = pd.concat(all_dfs, ignore_index=True)
summary_table = pd.DataFrame(all_summaries)
per_seed_results

In [ ]:
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
summary_table

In [ ]:
### Baseline: k-means στον αρχικό χώρο (768 διαστάσεις)

km_raw = KMeans(n_clusters=len(np.unique(labels)), random_state=0, n_init=10).fit(embeddings)
print(f"Raw 768-d: ARI = {adjusted_rand_score(labels, km_raw.labels_):.4f} | "
      f"NMI = {normalized_mutual_info_score(labels, km_raw.labels_):.4f}")

## 5. Οπτικοποίηση αποτελεσμάτων

In [ ]:
metrics_to_plot = ["trustworthiness", "continuity", "knn_overlap"]
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(15, 4))

for ax, metric in zip(axes, metrics_to_plot):
    per_seed_results.boxplot(column=metric, by="method", ax=ax)
    ax.set_title(metric)
    ax.set_xlabel("")

plt.suptitle("Διακύμανση μετρικών robustness ανά μέθοδο (5 seeds)")
plt.tight_layout()
plt.show()

In [ ]:
### Procrustes-aligned layouts ανά seed

for name, projs in all_projections.items():
    fig, axes = plt.subplots(1, len(SEED_LIST), figsize=(4 * len(SEED_LIST), 4))
    ref = projs[0]
    for ax, seed, proj in zip(axes, SEED_LIST, projs):
        _, m2, disparity = procrustes(ref, proj)
        for lab in np.unique(labels):
            idx = labels == lab
            ax.scatter(m2[idx, 0], m2[idx, 1], s=8, color=COLORS[lab], label=lab)
        ax.set_title(f"seed={seed}\ndisparity={disparity:.3f}")
    handles, names = axes[0].get_legend_handles_labels()
    fig.legend(handles, names, loc="lower center", ncol=len(names), markerscale=2)
    fig.suptitle(f"{name.upper()} — Procrustes-aligned layouts across seeds")
    fig.tight_layout(rect=(0, 0.08, 1, 0.93))
    plt.show()

###Sweep υπερπαραμέτρων UMAP και t-SNE

In [ ]:
HP_SEEDS = [0, 1, 2]

UMAP_DEFAULT = dict(n_neighbors=15, min_dist=0.1, metric="cosine")
TSNE_N_ITER = 1000

UMAP_N_NEIGHBORS_GRID = [5, 10, 15, 30, 50]
UMAP_MIN_DIST_GRID = [0.0, 0.1, 0.25, 0.5, 0.8]
UMAP_METRIC_GRID = ["cosine", "euclidean", "correlation"]
TSNE_PERPLEXITY_GRID = [5, 15, 30, 50, 100]


#UMAP (n_neighbors, min_dist, metric)
def run_umap_hp_sweep(embeddings, k=K_NEIGHBORS, seeds=HP_SEEDS):
    rows = []
    for param_name, grid in [("n_neighbors", UMAP_N_NEIGHBORS_GRID),
                              ("min_dist", UMAP_MIN_DIST_GRID),
                              ("metric", UMAP_METRIC_GRID)]:
        for value in grid:
            for seed in seeds:
                params = dict(UMAP_DEFAULT)
                params[param_name] = value
                reducer = umap.UMAP(n_components=2, random_state=seed, **params)
                emb_2d = reducer.fit_transform(embeddings)

                rows.append({
                    "swept_param": param_name, "value": value, "seed": seed,
                    "trustworthiness": trustworthiness(embeddings, emb_2d, n_neighbors=k),
                    "continuity": continuity(embeddings, emb_2d, k=k),
                    "knn_overlap": knn_overlap(embeddings, emb_2d, k=k),
                })
    return pd.DataFrame(rows)

#t-SNE (perplexity)
def run_tsne_hp_sweep(embeddings, k=K_NEIGHBORS, seeds=HP_SEEDS):
    rows = []
    for perplexity in TSNE_PERPLEXITY_GRID:
        for seed in seeds:
            reducer = TSNE(n_components=2, perplexity=perplexity, max_iter=TSNE_N_ITER,
                random_state=seed, init="pca")
            emb_2d = reducer.fit_transform(embeddings)

            rows.append({
                "swept_param": "perplexity", "value": perplexity, "seed": seed,
                "trustworthiness": trustworthiness(embeddings, emb_2d, n_neighbors=k),
                "continuity": continuity(embeddings, emb_2d, k=k),
                "knn_overlap": knn_overlap(embeddings, emb_2d, k=k),
            })
    return pd.DataFrame(rows)

In [ ]:
print("Εκτέλεση UMAP hyperparameter sweep...")
umap_hp_results = run_umap_hp_sweep(embeddings)

print("Εκτέλεση t-SNE perplexity sweep...")
tsne_hp_results = run_tsne_hp_sweep(embeddings)

# Σύνοψη (μέσος όρος / τυπική απόκλιση ανά τιμή, πάνω στα 3 seeds)
umap_hp_summary = umap_hp_results.groupby(["swept_param", "value"])[
    ["trustworthiness", "continuity", "knn_overlap"]].agg(["mean", "std"])
tsne_hp_summary = tsne_hp_results.groupby("value")[
    ["trustworthiness", "continuity", "knn_overlap"]].agg(["mean", "std"])

print("\n=== UMAP hyperparameter sweep ===")
print(umap_hp_summary)
print("\n=== t-SNE perplexity sweep ===")
print(tsne_hp_summary)

In [ ]:
# Οπτικοποίηση: trustworthiness συναρτήσει κάθε υπερπαραμέτρου
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, param in zip(axes, ["n_neighbors", "min_dist", "metric"]):
    sub = umap_hp_results[umap_hp_results["swept_param"] == param]
    sub.boxplot(column="trustworthiness", by="value", ax=ax)
    ax.set_title(f"UMAP: trustworthiness vs {param}")
    ax.set_xlabel(param)
plt.suptitle("")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
tsne_hp_results.boxplot(column="trustworthiness", by="value", ax=ax)
ax.set_title("t-SNE: trustworthiness vs perplexity")
ax.set_xlabel("perplexity")
plt.suptitle("")
plt.tight_layout()
plt.show()